# 03 — Match to Overture

This notebook matches labeled split edges from notebook 02 to Overture road
segments using densified point projection.  Outputs a matches table with
linear reference positions (LR 0–1) on each Overture segment.

In [ ]:
import os
import geopandas as gpd
from slc import fetch, match, viz

BBOX = (-111.920, 40.855, -111.855, 40.910)

In [ ]:
# Load split edges from notebook 02
split_edges = gpd.read_parquet('split_edges.parquet')
overture = fetch.fetch_overture_segments(BBOX)
print(f'Labeled edges: {split_edges["speed_mph"].notna().sum()}')

In [ ]:
# Match edges to Overture segments
matches = match.match_edges_to_overture(
    split_edges, overture,
    max_distance_m=25.0,
    min_overlap_fraction=0.3,
)
print(f'Matches: {len(matches)}')
matches.head()

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
matches['score'].hist(bins=20)
plt.xlabel('Match score')
plt.title('Distribution of match scores')
plt.show()

# LR range coverage
matches['lr_coverage'] = matches['lr_end'] - matches['lr_start']
print(f'Mean LR coverage per match: {matches["lr_coverage"].mean():.2f}')

In [ ]:
# How many Overture segments got at least one match?
covered = matches['overture_id'].nunique()
total_segs = len(overture)
print(f'Coverage: {covered}/{total_segs} = {covered/total_segs:.1%}')

In [ ]:
# Export matches
matches.to_parquet('matches.parquet', index=False)
print('Saved matches.parquet')